# 🏀 NBA Foul Detection & Prediction — Complete End-to-End Interactive Pipeline
**Course:** CSC 434 — Artificial Intelligence & Machine Learning Final Project  
**Architecture:** Cross-Platform Dual-Stream Spatiotemporal `R(2+1)D-18` 3D CNN + `YOLOv11-Pose` 12-D Kinematic Self-Attention Contact Encoder  

---
### ⚡ Google Colab & Jupyter Notebook Universal Quickstart
This self-contained notebook is engineered to run seamlessly across **Google Colab (Free T4 / Pro GPU)** and **Local Jupyter Notebook / Lab** on **macOS (Apple Silicon M1/M2/M3/M4 & Intel), Windows, and Linux (CUDA / CPU)**.

> [!TIP]
> **Running in Google Colab?**  
> Simply select **Runtime ➔ Change runtime type ➔ T4 GPU**, then run **Cell 1**. The universal bootstrap in Cell 1 will automatically clone the repository, install all dependencies (`ultralytics`, `opencv-python-headless`), and configure paths with zero manual setup!

| Stage | Module Equivalent | Core Responsibility | Lightweight Default |
| :--- | :--- | :--- | :--- |
| **Cell 1** | Universal Bootstrap & Config | Cross-OS & Colab path resolution, device auto-selection (`CUDA` / `MPS` / `CPU`), & knobs | `BATCH_SIZE = 4`, `ACCUM_STEPS = 2` |
| **Cell 2** | `1_scrape_l2m.py` | NBA Last Two Minute (L2M) scraper, UTF-8 BOM-safe parser, & ground-truth label mapper | `RUN_LIVE_WEB_SCRAPE = False` (offline sync) |
| **Cell 3** | Exploratory Data Analysis | Inspects `l2m_metadata.csv` (349 plays, 18+ games) with visual distribution charts | Instant Matplotlib summary |
| **Cell 4** | `2_preprocess_tubes.py` (Schema) | 12-D Mechanical Foul Dimensions grounded in NBA Rule 12 officiating | 12 Officiating Dimensions |
| **Cell 5** | `2_preprocess_tubes.py` (Runner) | Two-pass memory-safe action-tube extractor with offline synthetic `.pt` fallback | Test clip exclusion to prevent leakage |
| **Cell 6** | `3_model_and_dataset.py` | `L2MFoulDataset`, `BiomechanicalFoulEncoder` (4-head self-attention), & `FoulClassifier3D` | 512-D 3D CNN + 256-D Pose Fusion |
| **Cell 7** | `4_train_eval.py` (Train/Eval) | `FoulFocalBCEWithLogitsLoss` (`gamma=1.5`), `safe_foul_collate`, & dynamic F1 calibration | `EPOCHS = 10`, `FREEZE_BACKBONE = True` (~2s run) |
| **Cell 8** | `4_train_eval.py` (Inference) | Single-clip inference, 16-frame contact telemetry curves & 12-D mechanical radar/bar chart | Calibrated threshold verdict |
| **Cell 9** | `4_train_eval.py` (Video HUD) | Multimodal court player tracking, green-to-red box transition at foul frame & Top-Left HUD | Interactive HTML5 Video Player |


In [ ]:
# ==============================================================================
# CELL 1: Universal Bootstrap, Cross-Platform Environment Setup & Config
# ==============================================================================
import os
import sys
import csv
import gc
import html
import json
import time
import random
import hashlib
import logging
import platform
import subprocess
from pathlib import Path
from collections import Counter
from typing import Any, Callable, Dict, List, Optional, Set, Tuple

# ------------------------------------------------------------------------------
# 1. Google Colab Environment Auto-Bootstrap
# ------------------------------------------------------------------------------
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('🚀 Running in Google Colab environment!')
    print('📦 Installing required dependencies...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'opencv-python-headless', 'scikit-learn', 'matplotlib'], check=False)
    
    repo_dir = Path('/content/AI-ML-Final-Project')
    if not repo_dir.exists():
        print('📥 Cloning repository from GitHub (testing branch)...')
        subprocess.run(['git', 'clone', '-b', 'testing', 'https://github.com/ferraromike/AI-ML-Final-Project.git', str(repo_dir)], check=False)
    
    target_dir = repo_dir / 'NBA-Foul-Classifier-WIP'
    if target_dir.exists():
        os.chdir(str(target_dir))
    else:
        os.chdir(str(repo_dir))
    print(f'📁 Working directory set to: {Path.cwd()}')

# Prevent OpenMP / MKL / OpenCV FFmpeg thread deadlocks across Windows, macOS & Linux
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '2')
os.environ.setdefault('MKL_NUM_THREADS', '2')

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision.transforms.functional as TF
from torchvision.models.video import R2Plus1D_18_Weights, r2plus1d_18
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

cv2.setNumThreads(0)

try:
    from ultralytics import YOLO
except ImportError:
    YOLO = None

# ------------------------------------------------------------------------------
# 2. Automatic Project Directory & Weights Path Resolution
# ------------------------------------------------------------------------------
CWD = Path.cwd().resolve()
if (CWD / 'NBA-Foul-Classifier-WIP').exists():
    PROJECT_DIR = CWD / 'NBA-Foul-Classifier-WIP'
elif (CWD / '3_model_and_dataset.py').exists():
    PROJECT_DIR = CWD
elif (CWD.parent / 'NBA-Foul-Classifier-WIP').exists():
    PROJECT_DIR = CWD.parent / 'NBA-Foul-Classifier-WIP'
else:
    PROJECT_DIR = CWD

for p in [PROJECT_DIR, PROJECT_DIR.parent]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

DATA_DIR = PROJECT_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
METADATA_CSV = DATA_DIR / 'l2m_metadata.csv'
INCORRECT_CSV = DATA_DIR / 'l2m_incorrect_fouls.csv'
PROCESSED_METADATA_CSV = PROCESSED_DIR / 'processed_metadata.csv'
WEIGHTS_DIR = PROJECT_DIR / 'weights'
TEST_DIR = DATA_DIR / 'test'
ANNOTATED_DIR = DATA_DIR / 'annotated'

# Auto-discover trained checkpoint path
CHECKPOINT_PATH = WEIGHTS_DIR / 'best_foul_r2plus1d.pt'
if not CHECKPOINT_PATH.exists() and (PROJECT_DIR / 'best_foul_r2plus1d.pt').exists():
    CHECKPOINT_PATH = PROJECT_DIR / 'best_foul_r2plus1d.pt'

# ------------------------------------------------------------------------------
# 3. Device Selection & Hardware Acceleration
# ------------------------------------------------------------------------------
def select_compute_device(requested: Optional[str] = None) -> torch.device:
    if requested and requested.lower() != 'auto':
        return torch.device(requested)
    if torch.cuda.is_available():
        return torch.device('cuda')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

def get_safe_num_workers(requested_workers: int = 0) -> int:
    if platform.system() in ('Darwin', 'Windows'):
        return 0
    return max(0, requested_workers)

DEVICE = select_compute_device()
device_details = f'{DEVICE.type.upper()}'
if DEVICE.type == 'cuda':
    device_details += f' ({torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM)'
elif DEVICE.type == 'mps':
    device_details += f' (Apple Silicon GPU Acceleration)'
else:
    device_details += f' ({os.cpu_count()} CPU cores)'

print(f'🖥️  Platform         : {platform.system()} ({platform.machine()})')
print(f'🔥 PyTorch Version  : {torch.__version__}')
print(f'⚡ Compute Device   : {device_details}')
print(f'📁 Project Directory: {PROJECT_DIR}')

# ------------------------------------------------------------------------------
# 4. High-Performance / Zero-Stall Configuration Knobs
# ------------------------------------------------------------------------------
CONFIG = {
    'max_games': 3,              # Games to query if live scraping (Default: 3)
    'max_scrape_clips': 50,      # Max clips to download if scraping (Default: 50)
    'max_preprocess_clips': 50,  # Max clips to preprocess in Cell 5 (Default: 50)
    'max_live_play_sec': 30.0,   # Scan first 30s of clip, skipping post-play replays
    'max_scan_samples': 12,      # Pose scan frames per clip for peak ROI estimation
    'yolo_batch_size': 4,        # Micro-batch size for YOLOv11-Pose
    'num_frames': 16,            # 16-frame spatiotemporal action tube window
    'crop_size': 224,            # 224x224 spatial crop around active contact interaction
    'batch_size': 4,             # DataLoader micro-batch size (Scale to 8-16 on CUDA)
    'accum_steps': 2,            # Gradient accumulation steps (Effective batch = 8)
    'epochs': 10,                # Training epochs (Completes in ~2s with frozen backbone!)
    'max_train_samples': 0,      # 0 loads all 123 processed samples; set > 0 to limit
    'lr': 1.2e-4,                # Learning rate for AdamW
    'weight_decay': 1e-2,        # L2 weight decay regularization
    'dropout': 0.35,             # Dropout rate for classifier head
    'freeze_backbone': True,     # True trains 4-Head Attention Pose Network + MLP using cached embeddings (<3s runtime)
    'seed': 42,                  # Deterministic reproducibility seed
}


---
## Stage 1: NBA Last Two Minute (L2M) Scraper & Ground-Truth Labeler (`1_scrape_l2m.py`)
Official NBA Last Two Minute (L2M) reports provide transparent, post-game officiating evaluations for all contested plays in the final 2 minutes of close games (within 3 points). We map decisions to binary ground-truth labels:

- **Positive Class (`1 = foul`)**:
  - `CC` (*Correct Call*): Referee whistled a foul, and NBA league review confirmed illegal contact occurred.
  - `INC` (*Incorrect Non-Call*): Referee allowed play to continue, but league review confirmed a foul was committed (*crucial ground-truth signal*).
- **Negative Class (`0 = no_foul`)**:
  - `CNC` (*Correct Non-Call*): Legal defensive contest, no whistle blown.
  - `IC` (*Incorrect Call*): Referee blew the whistle, but league review found legal guarding positioning (*phantom foul*).

The cell below synchronizes `data/l2m_incorrect_fouls.csv` with the cached ground-truth metadata in `data/l2m_metadata.csv` (or queries `official.nba.com` if `RUN_LIVE_WEB_SCRAPE = True`).


In [ ]:
# ==============================================================================
# CELL 2: Stage 1 — NBA L2M Scraper & Local Metadata Sync
# ==============================================================================
import importlib
scrape_mod = importlib.import_module('1_scrape_l2m')

RUN_LIVE_WEB_SCRAPE = False  # Set to True to query official.nba.com for CONFIG['max_games'] games

if RUN_LIVE_WEB_SCRAPE:
    print(f'Scraping official NBA L2M reports for up to {CONFIG["max_games"]} games...')
    scraped_rows = scrape_mod.run_training_dataset_scraper(
        output_dir=RAW_DIR,
        metadata_csv=METADATA_CSV,
        daily_csv=INCORRECT_CSV,
        max_games=CONFIG['max_games'],
        max_clips=CONFIG['max_scrape_clips'],
        skip_download=False,
    )
    print(f'Scraped {len(scraped_rows)} plays into {METADATA_CSV.name}')
else:
    # Synchronize & populate l2m_incorrect_fouls.csv offline from cached l2m_metadata.csv
    populated_foul_rows = scrape_mod.populate_foul_csv_from_local_metadata(
        metadata_csv=METADATA_CSV,
        daily_csv=INCORRECT_CSV,
        include_confirmed_fouls=True,
    )
    print(f'✅ Successfully synced {len(populated_foul_rows)} foul review entries into {INCORRECT_CSV.name}')


---
## Exploratory Data Analysis (EDA): Inspecting the L2M Foul Dataset
Before training deep neural networks, we examine the class balance, review decisions (`CNC`, `CC`, `INC`, `IC`), and foul types (`Shooting Foul` vs. `Personal Foul`) curated in `data/l2m_metadata.csv`.


In [ ]:
# ==============================================================================
# CELL 3: Dataset Summary, Class Imbalance & Visual Distributions
# ==============================================================================
with open(METADATA_CSV, 'r', encoding='utf-8') as f:
    meta_rows = list(csv.DictReader(f))

label_counts = Counter(int(r['label']) for r in meta_rows)
decision_counts = Counter(r['review_decision'] for r in meta_rows)
call_counts = Counter(r['call_type'] for r in meta_rows)
unique_games = sorted(set(r['game_id'] for r in meta_rows))

print(f'📊 Total Labeled Plays : {len(meta_rows)} across {len(unique_games)} NBA games')
print(f'⚖️  Class Distribution  : no_foul (0) = {label_counts[0]} ({label_counts[0]/len(meta_rows)*100:.1f}%) | '
      f'foul (1) = {label_counts[1]} ({label_counts[1]/len(meta_rows)*100:.1f}%)')
print(f'📈 Imbalance Ratio     : {label_counts[0] / max(1, label_counts[1]):.2f} : 1 (no_foul : foul)')
print(f'📋 Review Decisions    : {dict(decision_counts)}')
print(f'🏀 Foul Call Types     : {dict(call_counts)}')

# ------------------------------------------------------------------------------
# Matplotlib EDA Visualizations
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Chart 1: Binary Class Distribution
bars = axes[0].bar(['Legal Play / No Foul (0)', 'Foul (1)'], [label_counts[0], label_counts[1]], color=['#2ca02c', '#d62728'], width=0.55)
axes[0].set_ylabel('Number of Plays', fontsize=11)
axes[0].set_title('Binary Class Distribution (L2M Ground Truth)', fontsize=12, fontweight='bold')
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., h + 5, f'{h} ({h/len(meta_rows)*100:.1f}%)', ha='center', va='bottom', fontweight='bold')
axes[0].set_ylim(0, max(label_counts.values()) * 1.15)
axes[0].grid(axis='y', alpha=0.3)

# Chart 2: Officiating Review Decisions
dec_keys = list(decision_counts.keys())
dec_vals = [decision_counts[k] for k in dec_keys]
colors = ['#2ca02c' if k == 'CNC' else '#d62728' if k == 'CC' else '#ff7f0e' if k == 'INC' else '#9467bd' for k in dec_keys]
axes[1].bar(dec_keys, dec_vals, color=colors, width=0.55)
axes[1].set_ylabel('Number of Reviews', fontsize=11)
axes[1].set_title('NBA Official Review Decision Breakdown', fontsize=12, fontweight='bold')
for i, v in enumerate(dec_vals):
    axes[1].text(i, v + 4, str(v), ha='center', va='bottom', fontweight='bold')
axes[1].set_ylim(0, max(dec_vals) * 1.15)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print('Sample disputed plays from l2m_metadata.csv:')
for sample_row in meta_rows[:3]:
    print(f'  • [{sample_row["play_id"]}] {sample_row["call_type"]} ({sample_row["review_decision"]} -> label={sample_row["label"]}): '
          f'{sample_row["player_committing"]} on {sample_row["player_disadvantaged"]}')


---
## Stage 2: Biomechanical Pose Estimation & Action-Tube Extraction (`2_preprocess_tubes.py`)
Feeding uncompressed 1080p broadcast video clips directly into 3D Convolutional Neural Networks leads to **RAM/VRAM overflow** (L2M clips often exceed 10,000 frames / 76+ GB uncompressed) and **spatial dilution** (interacting players occupy < 8% of the court pixels).

### Our Two-Pass Cross-Platform Solution:
1. **Pass 1 (Lightweight Pose Scan)**: Scans the live play window using `yolo11n-pose.pt` in micro-batches (`yolo_batch_size = 4`). It extracts a **12-D Mechanical Foul Feature Vector** grounded in NBA Rule 12 officiating rules.
2. **Court Geometry & Sideline Rejection**: Strictly filters court floor coordinates ($y_c \in [0.22H, 0.68H]$, $y_2 \le 0.72H$) to reject referees (#43), bench players, and background spectators.
3. **Pass 2 (Stabilized 16-Frame Crop)**: Applies Exponential Moving Average (EMA) smoothing to the bounding box trajectory and crops **only the 16 frames** in a `1.5s` window around peak contact at `224x224` resolution, saving `(3, 16, 224, 224)` RGB tubes and `(16, 12)` pose tensors.
4. **Data Isolation**: All clips residing in `data/test/` are strictly excluded from training preprocessing to prevent data leakage.


In [ ]:
# ==============================================================================
# CELL 4: Inspect Stage 2 Mechanical Foul Dimensions Schema
# ==============================================================================
preprocess_mod = importlib.import_module('2_preprocess_tubes')
importlib.reload(preprocess_mod)

MECHANICAL_FOUL_DESCRIPTIONS = [
    ('0: arm_contact_intrusion', 'Defender hand/wrist collision with shooter arm during gather/release'),
    ('1: body_cylinder_penetration', 'Defender torso/hip invasion into offensive cylinder violating legal guarding'),
    ('2: lower_body_landing_violation', 'Defender feet/legs occupying shooter landing space (Zaza Pachulia rule)'),
    ('3: defender_verticality_angle', 'Angle deviation from legal 90° vertical contest (Rule 12B Section I)'),
    ('4: defender_lateral_positioning', 'Defender lateral offset relative to offensive drive path (blocking vs charge)'),
    ('5: offensive_displacement', 'Shooter trajectory/balance shift resulting from physical collision'),
    ('6: defender_to_ball_distance', 'Distance from defender hand to basketball (swiping accuracy)'),
    ('7: ball_to_shooter_distance', 'Proximity of basketball to shooter (gather vs release state)'),
    ('8: ball_strike_vs_body_ratio', 'Hand-to-ball distance vs. hand-to-body distance (clean block vs hack)'),
    ('9: relative_closing_velocity', 'Speed at which defender closes space toward the shooter'),
    ('10: momentum_transfer_shock', 'Sudden deceleration/jolt upon impact indicating physical force transfer'),
    ('11: airborne_contest_apex_diff', 'Vertical contest differential between shooter and defender at apex'),
]

print('🏀 12 Mechanical Foul Dimensions Grounded in NBA Rulebook Officiating:')
for dim_name, dim_desc in MECHANICAL_FOUL_DESCRIPTIONS:
    print(f'  • {dim_name:<34} : {dim_desc}')


In [ ]:
# ==============================================================================
# CELL 5: Run Stage 2 Preprocessing & Dataset Validation
# ==============================================================================
# Preprocesses balanced clips excluding data/test/ to prevent data leakage.
# If raw MP4s are present, uses YOLOv11-Pose; otherwise leverages precomputed/synthetic fallback.
import inspect

pre_sig = inspect.signature(preprocess_mod.preprocess_dataset)
pre_kwargs = {
    'metadata_csv': METADATA_CSV,
    'processed_dir': PROCESSED_DIR,
    'processed_metadata_csv': PROCESSED_METADATA_CSV,
    'window_sec': 1.5,
    'num_frames': CONFIG['num_frames'],
    'crop_size': CONFIG['crop_size'],
    'device': str(DEVICE),
    'max_live_play_sec': CONFIG['max_live_play_sec'],
    'max_scan_samples': CONFIG['max_scan_samples'],
    'max_clips': CONFIG['max_preprocess_clips'],
    'yolo_batch_size': CONFIG['yolo_batch_size'],
    'synthetic_fallback': True,
}
# Backward compatibility: dynamically filter kwargs so older and newer module versions run without TypeError
if 'test_dir' in pre_sig.parameters:
    pre_kwargs['test_dir'] = TEST_DIR
if 'pose_weights' in pre_sig.parameters and (WEIGHTS_DIR / 'yolo11n-pose.pt').exists():
    pre_kwargs['pose_weights'] = str(WEIGHTS_DIR / 'yolo11n-pose.pt')
if 'yolo_weights' in pre_sig.parameters and (WEIGHTS_DIR / 'yolo11n.pt').exists():
    pre_kwargs['yolo_weights'] = str(WEIGHTS_DIR / 'yolo11n.pt')

valid_pre_kwargs = {k: v for k, v in pre_kwargs.items() if k in pre_sig.parameters}
processed_rows = preprocess_mod.preprocess_dataset(**valid_pre_kwargs)

# Check count of preprocessed tensors in data/processed/
foul_pts = list((PROCESSED_DIR / 'foul').glob('*.pt'))
no_foul_pts = list((PROCESSED_DIR / 'no_foul').glob('*.pt'))
total_tensors = len(foul_pts) + len(no_foul_pts)
print(f'✅ Processed Dataset Status : {total_tensors} total action tubes ready')
print(f'   • Foul action tubes      : {len(foul_pts)}')
print(f'   • No-Foul action tubes   : {len(no_foul_pts)}')
if PROCESSED_METADATA_CSV.exists():
    print(f'   • Processed metadata     : {PROCESSED_METADATA_CSV.name}')


---
## Stage 3: Dual-Stream Spatiotemporal Architecture (`3_model_and_dataset.py`)
Our classifier fuses two complementary representations into a unified 768-D representation:

1. **Visual Spatiotemporal Stream (`R(2+1)D-18`)**: Decomposes 3D convolutions into $2D$ spatial $+ 1D$ temporal operations over the `(3, 16, 224, 224)` action tube, yielding a **512-D visual embedding**.
2. **Kinematic Temporal Attention Pose Stream (`BiomechanicalFoulEncoder`)**: Projects the `(16, 12)` mechanical foul sequence into a 128-D space, applies **4-Head Temporal Self-Attention** across all 16 frames with residual connection and LayerNorm, followed by dual temporal pooling (mean + max) to produce a **256-D kinematic contact embedding**.
3. **Late-Fusion Classification Head**: Concatenates `[512-D Visual || 256-D Pose] = 768-D` and passes through a 3-layer MLP (`768 -> 512 -> 256 -> 1`) with LayerNorm, GELU, and Dropout.

> [!NOTE]
> **Zero-Stall Training Architecture:** Freezing the 3D ConvNet backbone and utilizing precomputed 512-D Kinetics embeddings allows the 4-head attention network and fusion MLP (~613K parameters) to train in **under 3 seconds** on laptops and Google Colab, with zero swap thrashing or memory bottlenecks!


In [ ]:
# ==============================================================================
# CELL 6: Verify Dataset Loading, Collation & Dual-Stream Forward Pass
# ==============================================================================
import inspect
model_mod = importlib.import_module('3_model_and_dataset')
importlib.reload(model_mod)

L2MFoulDataset = model_mod.L2MFoulDataset
FoulClassifier3D = model_mod.FoulClassifier3D

ds_sig = inspect.signature(model_mod.L2MFoulDataset.__init__)
ds_kwargs = {
    'processed_dir': PROCESSED_DIR,
    'metadata_csv': PROCESSED_METADATA_CSV if PROCESSED_METADATA_CSV.exists() else None,
    'is_train': True,
    'max_samples': CONFIG['max_train_samples'],
    'return_foul_features': True,
}
if 'test_dir' in ds_sig.parameters:
    ds_kwargs['test_dir'] = TEST_DIR
if 'use_cached_embeddings' in ds_sig.parameters:
    ds_kwargs['use_cached_embeddings'] = True

valid_ds_kwargs = {k: v for k, v in ds_kwargs.items() if k in ds_sig.parameters}
demo_dataset = L2MFoulDataset(**valid_ds_kwargs)

sample_video, sample_feats, sample_label = demo_dataset[0]
print(f'✅ Loaded Dataset Size   : {len(demo_dataset)} samples')
if sample_video.dim() == 1:
    print(f'📹 Video Representation  : shape={tuple(sample_video.shape)} (512-D cached Kinetics visual embedding)')
else:
    print(f'📹 Video Representation  : shape={tuple(sample_video.shape)} (3D action tube volume)')
print(f'🦴 Kinematic Features    : shape={tuple(sample_feats.shape)} (16 frames × {sample_feats.shape[-1]} mechanical dimensions)')
gt_label_str = 'FOUL' if sample_label.item() == 1 else 'NO FOUL'
print(f'🏷️  Ground-Truth Label    : {int(sample_label.item())} ({gt_label_str})')

# Instantiate model with kwargs dynamically matched to class signature
m_sig = inspect.signature(model_mod.FoulClassifier3D.__init__)
m_kwargs = {
    'pretrained': False,
    'dropout_p': CONFIG['dropout'],
}
if 'freeze_backbone' in m_sig.parameters:
    m_kwargs['freeze_backbone'] = CONFIG['freeze_backbone']
elif 'freeze_early' in m_sig.parameters:
    m_kwargs['freeze_early'] = True

valid_m_kwargs = {k: v for k, v in m_kwargs.items() if k in m_sig.parameters}
test_model = FoulClassifier3D(**valid_m_kwargs).to(DEVICE)

if hasattr(test_model, 'trainable_parameter_summary'):
    param_summary = test_model.trainable_parameter_summary()
else:
    tot = sum(p.numel() for p in test_model.parameters())
    trn = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
    param_summary = {'total_params': tot, 'trainable_params': trn, 'frozen_params': tot - trn}

with torch.no_grad():
    sample_logit = test_model(
        sample_video.unsqueeze(0).to(DEVICE),
        sample_feats.unsqueeze(0).to(DEVICE),
    )
    sample_prob = torch.sigmoid(sample_logit).item()

print(f'\nModel Architecture Parameter Summary:')
print(f'  • Total Parameters     : {param_summary["total_params"]:,}')
print(f'  • Trainable Parameters : {param_summary["trainable_params"]:,}')
print(f'  • Frozen Parameters    : {param_summary["frozen_params"]:,}')
print(f'  • Verification Forward : Logit = {sample_logit.item():.4f} -> P(foul) = {sample_prob*100:.2f}%')


---
## Stage 4: High-Performance Training Loop, Focal Loss & Threshold Calibration (`4_train_eval.py`)
To handle class imbalance and prevent PyTorch collation stalls:

1. **`safe_foul_collate`**: Dynamically verifies batch tensor shapes and harmonizes representations before `torch.stack`, completely eliminating PyTorch DataLoader `RuntimeError: stack expects each tensor to be equal size`.
2. **Class-Balanced `FoulFocalBCEWithLogitsLoss` (`gamma = 1.5`)**: Multiplies positive examples by $pos\_weight$ and dynamically scales loss by $(1 - p_t)^\gamma$ to penalize hard, subtle foul misclassifications.
3. **Dynamic F1 Threshold Calibration**: Sweeps probability thresholds $\tau \in [0.20, 0.80]$ on validation predictions at every epoch to maximize F1-score rather than assuming a naive $0.50$ cutoff.


In [ ]:
# ==============================================================================
# CELL 7: Train & Calibrate the Dual-Stream Foul Classifier
# ==============================================================================
import inspect
train_mod = importlib.import_module('4_train_eval')
importlib.reload(train_mod)

train_mod.seed_everything(CONFIG['seed'])

all_samples = demo_dataset.samples
labels = [lbl for _, lbl in all_samples]
stratify_labels = labels if len(set(labels)) > 1 and min(labels.count(0), labels.count(1)) >= 2 else None

train_samples, val_samples = train_test_split(
    all_samples,
    test_size=0.25,
    random_state=CONFIG['seed'],
    stratify=stratify_labels,
)

train_ds_kwargs = {'samples': train_samples, 'is_train': True, 'return_foul_features': True, 'use_cached_embeddings': True}
valid_train_ds_kwargs = {k: v for k, v in train_ds_kwargs.items() if k in ds_sig.parameters}
train_ds = L2MFoulDataset(**valid_train_ds_kwargs)

val_ds_kwargs = {'samples': val_samples, 'is_train': False, 'return_foul_features': True, 'use_cached_embeddings': True}
valid_val_ds_kwargs = {k: v for k, v in val_ds_kwargs.items() if k in ds_sig.parameters}
val_ds = L2MFoulDataset(**valid_val_ds_kwargs)

safe_workers = get_safe_num_workers(0)
collate_fn = getattr(train_mod, 'safe_foul_collate', None)
train_loader_kwargs = {'batch_size': CONFIG['batch_size'], 'shuffle': True, 'num_workers': safe_workers}
if collate_fn is not None:
    train_loader_kwargs['collate_fn'] = collate_fn
train_loader = DataLoader(train_ds, **train_loader_kwargs)

val_loader_kwargs = {'batch_size': CONFIG['batch_size'], 'shuffle': False, 'num_workers': safe_workers}
if collate_fn is not None:
    val_loader_kwargs['collate_fn'] = collate_fn
val_loader = DataLoader(val_ds, **val_loader_kwargs)

# Initialize classifier with kwargs dynamically matched to class signature
model = FoulClassifier3D(**valid_m_kwargs).to(DEVICE)

pos_weight = train_mod.compute_pos_weight(train_samples, device=DEVICE)
criterion = train_mod.FoulFocalBCEWithLogitsLoss(pos_weight=pos_weight, gamma=1.5)
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=max(1, CONFIG['epochs']), eta_min=CONFIG['lr'] * 0.01)
scaler = torch.amp.GradScaler('cuda', enabled=True) if (DEVICE.type == 'cuda' and torch.cuda.is_available()) else None

history = []
best_f1 = -1.0
start_train_time = time.time()

print(f'Starting training for {CONFIG["epochs"]} epochs on {DEVICE}...')
for epoch in range(1, CONFIG['epochs'] + 1):
    t_loss = train_mod.train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=DEVICE,
        accum_steps=CONFIG['accum_steps'],
    )
    v_loss, metrics = train_mod.evaluate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE,
    )
    scheduler.step()
    history.append({'epoch': epoch, 'train_loss': t_loss, 'val_loss': v_loss, **metrics})
    print(
        f'Epoch [{epoch:02d}/{CONFIG["epochs"]:02d}] | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | '
        f'Acc: {metrics["accuracy"]:.3f} | F1: {metrics["f1"]:.3f} | ROC-AUC: {metrics["roc_auc"]:.3f} | '
        f'Opt Thresh: {metrics["threshold"]:.2f}'
    )
    if metrics['f1'] >= best_f1:
        best_f1 = metrics['f1']
        WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
        torch.save(
            {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'metrics': metrics,
                'optimal_threshold': metrics['threshold'],
            },
            CHECKPOINT_PATH,
        )

elapsed = time.time() - start_train_time
print(f'\n⚡ Training completed in {elapsed:.2f}s! Best Calibrated F1-Score: {best_f1:.3f}')
print(f'💾 Saved calibrated model checkpoint -> {CHECKPOINT_PATH}')

# Matplotlib Training History Curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
epochs_range = [h['epoch'] for h in history]

axes[0].plot(epochs_range, [h['train_loss'] for h in history], 'o-', label='Train Focal Loss', color='#1f77b4', linewidth=2)
axes[0].plot(epochs_range, [h['val_loss'] for h in history], 's--', label='Val Focal Loss', color='#ff7f0e', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Focal Loss', fontsize=11)
axes[0].set_title('Training & Validation Loss Convergence', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, [h['accuracy'] for h in history], '^-', label='Accuracy', color='#2ca02c', linewidth=2)
axes[1].plot(epochs_range, [h['f1'] for h in history], 'o-', label='F1-Score', color='#d62728', linewidth=2)
axes[1].plot(epochs_range, [h['roc_auc'] for h in history], 'x--', label='ROC-AUC', color='#9467bd', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Metric Score (0 - 1)', fontsize=11)
axes[1].set_title('Validation Performance & F1 Calibration', fontsize=12, fontweight='bold')
axes[1].set_ylim(0.0, 1.05)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## Stage 5: Single-Clip Foul Prediction & Biomechanical Contact Telemetry
The cell below loads the calibrated `best_foul_r2plus1d.pt` checkpoint, evaluates an unseen play from the validation split, identifies the exact moment of peak physical impact across the 16-frame spatiotemporal tube, and displays:

1. **Continuous 16-Frame Contact Trajectory Plot**: Visualizes the evolution of arm intrusion, cylinder penetration, and relative closing velocity across time, marking the exact peak collision frame.
2. **12-D Mechanical Foul Dimension Profile**: Horizontal bar chart illustrating the complete biomechanical signature at the critical contact moment.


In [ ]:
# ==============================================================================
# CELL 8: Inference & Frame-by-Frame Biomechanical Telemetry Inspection
# ==============================================================================
ckpt = model_mod.load_torch_file_safe(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'], strict=False)
calibrated_threshold = float(ckpt.get('optimal_threshold', 0.50))
model.eval()

val_path, val_true_label = val_samples[0]
val_video, val_feats, _ = val_ds[0]

with torch.no_grad():
    logit = model(val_video.unsqueeze(0).to(DEVICE), val_feats.unsqueeze(0).to(DEVICE))
    foul_prob = float(torch.sigmoid(logit).item())

verdict = '🚨 FOUL DETECTED' if foul_prob >= calibrated_threshold else '✅ LEGAL PLAY / NO FOUL'
feats_np = val_feats.cpu().numpy()  # (16, 12)
contact_composite = feats_np[:, 0] * 0.40 + feats_np[:, 1] * 0.35 + feats_np[:, 9] * 0.25
peak_frame_idx = int(np.argmax(contact_composite))
peak_contact_score = float(contact_composite[peak_frame_idx])

print(f'🎯 Evaluated Play ID      : {val_path.stem}')
val_label_str = 'FOUL' if val_true_label == 1 else 'NO FOUL'
print(f'🏷️  Ground-Truth Label     : {val_true_label} ({val_label_str})')
print(f'📊 Predicted Probability   : {foul_prob * 100:.2f}% (Calibrated Threshold: {calibrated_threshold * 100:.0f}%)')
print(f'⚖️  Model Decision         : {verdict}')
print(f'⏱️  Peak Contact Frame     : Frame {peak_frame_idx + 1}/16 (Biomechanics Index = {peak_contact_score * 100:.1f}%)')

# ------------------------------------------------------------------------------
# Matplotlib Biomechanical Contact Telemetry Visualizations
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
frames_axis = list(range(1, 17))

# Left: 16-Frame Temporal Trajectory
axes[0].plot(frames_axis, feats_np[:, 0], 'o-', label='Arm Contact Intrusion', color='#d62728', linewidth=2)
axes[0].plot(frames_axis, feats_np[:, 1], 's-', label='Body Cylinder Penetration', color='#ff7f0e', linewidth=2)
axes[0].plot(frames_axis, feats_np[:, 9], '^--', label='Closing Velocity', color='#1f77b4', linewidth=1.8)
axes[0].plot(frames_axis, contact_composite, 'd:', label='Composite Contact Force', color='#9467bd', linewidth=2.2)
axes[0].axvline(x=peak_frame_idx + 1, color='crimson', linestyle='--', linewidth=2, label=f'Peak Contact (F{peak_frame_idx+1})')
axes[0].set_xlabel('Spatiotemporal Action Frame (1 - 16)', fontsize=11)
axes[0].set_ylabel('Normalized Mechanical Intensity (0 - 1)', fontsize=11)
axes[0].set_title('Biomechanical Contact Trajectory Across Time', fontsize=12, fontweight='bold')
axes[0].set_ylim(-0.05, 1.05)
axes[0].legend(loc='upper left', fontsize=9.5)
axes[0].grid(True, alpha=0.3)

# Right: 12-D Mechanical Dimensions Profile at Peak Contact Frame
dim_labels = list(model_mod.POSE_FEATURE_NAMES)
peak_magnitudes = feats_np[peak_frame_idx]
bar_colors = ['#d62728' if v > 0.45 else '#1f77b4' for v in peak_magnitudes]
axes[1].barh(dim_labels, peak_magnitudes, color=bar_colors, height=0.65)
axes[1].set_xlabel('Normalized Magnitude (0 - 1)', fontsize=11)
axes[1].set_title(f'12-D Mechanical Foul Signature at Peak Frame {peak_frame_idx+1}', fontsize=12, fontweight='bold')
axes[1].set_xlim(0, 1.05)
axes[1].grid(axis='x', alpha=0.3)
for i, v in enumerate(peak_magnitudes):
    if v > 0.05:
        axes[1].text(v + 0.02, i, f'{v:.2f}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


---
## Stage 6: Continuous Multimodal Video Annotation & Test Inference (`4_train_eval.py`)
Our inference engine runs full-duration multimodal tracking across all broadcast frames:

- **Court Geometry Filtering**: Completely eliminates sideline officials (e.g. Referee #43), bench players, and baseline spectators.
- **Dynamic Color Transition**: The bounding box tracks the disputed contest in **GREEN** (`LEGAL PLAY | NO FOUL`) until the exact foul frame $t_{\text{foul}}$, where physical contact surges into an illegal breach and transitions to bright **RED** (`FOUL DETECTED`).
- **Top-Left HUD Panel**: Explicitly displays **WHAT FOUL** occurred and **WHO FOULED WHOM** (e.g. `Shooting Foul: Dalen Terry fouled Ricky Council IV`).
- **Interactive Video Player**: Drop any `.mp4` into `data/test/` to view continuously tracked results directly within this notebook!


In [ ]:
# ==============================================================================
# CELL 9: Continuous Video Annotation on Dedicated Test Clips (data/test/)
# ==============================================================================
import inspect
from IPython.display import HTML, display

test_clips = sorted(list(TEST_DIR.glob('*.mp4'))) if TEST_DIR.exists() else []
if not test_clips and RAW_DIR.exists():
    test_clips = sorted(list(RAW_DIR.rglob('*.mp4')))
print(f'Found {len(test_clips)} clip(s) for inference: {[c.name for c in test_clips[:4]]}')

if test_clips and (YOLO is not None):
    sample_test_clip = test_clips[0]
    annotated_output_path = ANNOTATED_DIR / f'{sample_test_clip.stem}_annotated.mp4'
    ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)
    
    pred_sig = inspect.signature(train_mod.predict_single_clip)
    pred_kwargs = {
        'video_path': sample_test_clip,
        'checkpoint_path': CHECKPOINT_PATH,
        'output_mp4_path': annotated_output_path,
        'yolo_weights': str(WEIGHTS_DIR / 'yolo11n.pt') if (WEIGHTS_DIR / 'yolo11n.pt').exists() else 'yolo11n.pt',
        'pose_weights': str(WEIGHTS_DIR / 'yolo11n-pose.pt') if (WEIGHTS_DIR / 'yolo11n-pose.pt').exists() else 'yolo11n-pose.pt',
    }
    valid_pred_kwargs = {k: v for k, v in pred_kwargs.items() if k in pred_sig.parameters}
    
    print(f'\n🎬 Running end-to-end multimodal tracking & HUD annotation on {sample_test_clip.name}...')
    foul_score = train_mod.predict_single_clip(**valid_pred_kwargs)
    
    print(f'\n✅ Successfully generated annotated video!')
    print(f'   • Output Path: {annotated_output_path}')
    if foul_score is not None:
        print(f'   • Predicted Foul Probability: {foul_score * 100:.2f}%')
    
    if annotated_output_path.exists():
        rel_path = os.path.relpath(annotated_output_path, Path.cwd())
        html_markup = (
            '<div style="text-align: center; margin-top: 15px;">'
            '<h4>📹 Continuous Tracked Video Prediction Preview</h4>'
            f'<video width="680" height="380" controls autoplay muted loop style="border-radius: 8px; border: 2px solid #444;">'
            f'<source src="{rel_path}" type="video/mp4">'
            'Your browser does not support the video tag.'
            '</video>'
            f'<p style="font-size: 12px; color: #666;">File: {annotated_output_path.name}</p>'
            '</div>'
        )
        display(HTML(html_markup))
else:
    print('Tip: Place any raw .mp4 clip into data/test/ and rerun this cell to produce annotated output!')
    print('You can also run batch annotation on all test clips via command line:')
    print('python 4_train_eval.py --annotate-test')
